In [ ]:
import collections, sys, warnings
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
np.set_printoptions(precision=6, suppress=True, linewidth=120)

REPO = Path.cwd()
while not (REPO / 'data_fp32').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / 'data_fp32').is_dir(), 'could not locate the repo root (no data_fp32/)'

DATA   = REPO / 'data_fp32'          # float32 weights. NEVER data/, that one is int16.
BATCHD = REPO / 'aieml_batch'
SIMOUT = BATCHD / 'results' / 'sim_events'
SYSD   = BATCHD / 'sysdata'          # the RTP payloads the XRT host loads
TESTD  = REPO / 'testdata'
OUTD   = BATCHD / 'notebooks' / 'out'
OUTD.mkdir(parents=True, exist_ok=True)

# Geometry - must match the compiled graph (src/parameters.h, kernels/track_accum.h)
H                = 128    # HIDDEN_SIZE
TRACKS_PER_EVENT = 50     # TA_EVENT
SLOTS_PER_EVENT  = 56     # 7 iterations x BATCH 8
BATCH            = 8
OUT_DIM          = 27
REAL_FEATURES    = 6      # physically meaningful inputs; the files carry 8 (zero-padded)
WARMUP           = 3      # receptive field is 4 tracks deep -> first 3 are contaminated

print('repo   :', REPO)
print('numpy', np.__version__, '| onnx', onnx.__version__, '| onnxruntime', ort.__version__)


In [ ]:
ONNX_PATH = REPO / 'punit/onnx_no-residual/onnx_files_narrow/mlp_fp32.onnx'
assert ONNX_PATH.exists(), ONNX_PATH

def load_txt(name):
    return np.loadtxt(DATA / name).astype(np.float32)

def load_parts(stem, n, rows, cols):
    a = np.concatenate([load_txt(f'{stem}_part{i}.txt') for i in range(n)])
    return a.reshape(rows, cols).astype(np.float32)

# Every dense layer, exactly as gen_graph.py assembles it for the AIE.
# emb_d0 is stored 8x128 on disk but the model is 6->128: columns 6,7 are zero pad.
W = {'emb_d0': load_txt('embed_dense_0_weights.txt').reshape(8, H)[:REAL_FEATURES],
     'emb_d1': load_parts('embed_dense_1_weights', 2, H, H)}
B = {'emb_d0': load_txt('embed_dense_0_bias.txt'),
     'emb_d1': load_txt('embed_dense_1_bias.txt')}
for s in range(3):
    W[f's{s}_d0'] = load_parts(f'solver_{s}_dense_0_weights', 4, 2 * H, H)
    B[f's{s}_d0'] = load_txt(f'solver_{s}_dense_0_bias.txt')
    for d in (1, 2, 3):
        W[f's{s}_d{d}'] = load_parts(f'solver_{s}_dense_{d}_weights', 2, H, H)
        B[f's{s}_d{d}'] = load_txt(f'solver_{s}_dense_{d}_bias.txt')
W['output'] = load_txt('output_weights.txt').reshape(H, OUT_DIM)
B['output'] = load_txt('output_bias.txt')

model = onnx.load(ONNX_PATH)
inits = {i.name: onnx.numpy_helper.to_array(i) for i in model.graph.initializer}

def closest(ref, ndim):
    best = np.inf
    for a in inits.values():
        if a.ndim != ndim or a.size != ref.size:
            continue
        for cand in ((a, a.T) if ndim == 2 else (a,)):
            if cand.shape == ref.shape:
                best = min(best, float(np.abs(cand.astype(np.float32) - ref).max()))
    return best

print(f'[link 1] {ONNX_PATH.relative_to(REPO)}  vs  data_fp32/\n')
wq = {n: closest(r, 2) for n, r in W.items()}
bq = {n: closest(r, 1) for n, r in B.items()}
for n in W:
    print(f'  {n:9s} W {wq[n]:.3e}   b {bq[n]:.3e}')
LINK1 = max(max(wq.values()), max(bq.values()))
print(f'\n  worst over {len(wq)} weights + {len(bq)} biases: {LINK1:.3e}')
assert LINK1 < 1e-6, 'this ONNX is a DIFFERENT checkpoint - do not use it as the reference'
print('  PASS - the ONNX carries the data_fp32 weights.')


In [ ]:
def pack_mmul(Wm, K, N, K_slice, N_slice, mk, mn, cas_length, cas_num):
    tk, tn = K_slice // mk, N_slice // mn
    out = np.zeros((cas_num, cas_length, tk * tn * mk * mn), np.float32)
    for c in range(cas_num):
        n_base = c * N_slice
        for s in range(cas_length):
            i = 0
            for kt in range(tk):
                gk = s * K_slice + kt * mk
                for nt in range(tn):
                    tile = np.zeros((mk, mn), np.float32)
                    gn = n_base + nt * mn
                    rk, rn = max(0, min(mk, K - gk)), max(0, min(mn, N - gn))
                    if rk > 0 and rn > 0:
                        tile[:rk, :rn] = Wm[gk:gk + rk, gn:gn + rn]
                    out[c, s, i * mk * mn:(i + 1) * mk * mn] = tile.ravel('C')
                    i += 1
    return out

manifest_path = SYSD / 'rtp_manifest.txt'
if not manifest_path.exists():
    print(f'[link 2] SKIPPED - {manifest_path} not found.')
    print('         Produce it with:  cd aieml_batch && make system_graph && make rtp')
    LINK2 = None
else:
    # emb_d0 is presented 16 wide to the graph (INPUT_DIM=16), zero-padded from 8.
    W0 = np.zeros((16, H), np.float32)
    W0[:8] = load_txt('embed_dense_0_weights.txt').reshape(8, H)
    LAYERS = [('emb_d0', W0, B['emb_d0'], 16, 1, 1),
              ('emb_d1', W['emb_d1'], B['emb_d1'], H, 2, 2)]
    for s in range(3):
        LAYERS.append((f's{s}_d0', W[f's{s}_d0'], B[f's{s}_d0'], 2 * H, 4, 2))
        for d in (1, 2, 3):
            LAYERS.append((f's{s}_d{d}', W[f's{s}_d{d}'], B[f's{s}_d{d}'], H, 2, 2))

    entries = [l.split() for l in manifest_path.read_text().splitlines()
               if l.strip() and not l.startswith('#')]
    by_port = {p: f for p, _, f in entries}

    worst_w = worst_b = 0.0
    n_w = n_b = 0
    missing = []
    for name, Wm, Bv, K, cl, cn in LAYERS:
        Ks, Ns = K // cl, H // cn
        packed = pack_mmul(Wm, K, H, Ks, Ns, 8, 4, cl, cn)
        for c in range(cn):
            for s in range(cl):
                port = f'dut.dut.{name}_aie.kk[{c * cl + s}].in[1]'
                if port not in by_port:
                    missing.append(port); continue
                got = np.fromfile(SYSD / by_port[port], np.float32)
                worst_w = max(worst_w, float(np.abs(got - packed[c, s]).max())); n_w += 1
            bport = f'dut.dut.{name}_aie.kk[{(c + 1) * cl - 1}].in[{2 if cl == 1 else 3}]'
            if bport not in by_port:
                missing.append(bport); continue
            got = np.fromfile(SYSD / by_port[bport], np.float32)
            worst_b = max(worst_b, float(np.abs(got - Bv[c * Ns:(c + 1) * Ns]).max())); n_b += 1

    LINK2 = max(worst_w, worst_b)
    print(f'[link 2] data_fp32/  vs  the {len(entries)} RTP payloads in {SYSD.name}/\n')
    print(f'  weight ports : {n_w:3d}   worst max|diff| = {worst_w:.3e}')
    print(f'  bias   ports : {n_b:3d}   worst max|diff| = {worst_b:.3e}')
    print(f'  unmatched port names: {missing or "none"}')
    assert not missing and LINK2 == 0.0, 'the AIE is not running the data_fp32 weights'
    print(f'\n  PASS - all {n_w + n_b} payloads are BIT-EXACT.')
    print('\n  Chain closed: ONNX == data_fp32 == what the AIE multiplies.')


In [ ]:
g = model.graph
shp = lambda v: [d.dim_value or d.dim_param for d in v.type.tensor_type.shape.dim]
print('inputs  :', [(v.name, shp(v)) for v in g.input])
print('outputs :', [(v.name, shp(v)) for v in g.output])
print('ops     :', dict(collections.Counter(n.op_type for n in g.node)))

rm = next(n for n in g.node if n.op_type == 'ReduceMean')
PER_TRACK_T = rm.input[0]     # (batch, 50, 128) - every track's final activation
MEAN_T      = rm.output[0]    # (batch, 128)     - what the AIE emits as track_out
print('\nper-track tensor :', PER_TRACK_T)
print('event-mean tensor:', MEAN_T)

def make_session(path, extra_outputs=()):
    # The model only declares `output` (27 values). Adding the intermediates to the graph
    # outputs changes no arithmetic - it stops onnxruntime optimising them away.
    m = onnx.load(path)
    have = {v.name for v in m.graph.output}
    for t in extra_outputs:
        if t not in have:
            m.graph.output.append(onnx.helper.ValueInfoProto(name=t))
    return ort.InferenceSession(m.SerializeToString(), providers=['CPUExecutionProvider'])

SESS = make_session(ONNX_PATH, (PER_TRACK_T, MEAN_T))
OUT_NAMES = [o.name for o in SESS.get_outputs()]

def run_onnx(tracks6, chunk=100):
    # tracks6: (n_events, 50, 6) -> dict with per_track / mean / out27.
    tracks6 = np.asarray(tracks6, dtype=np.float32)
    acc = collections.defaultdict(list)
    for i in range(0, len(tracks6), chunk):
        res = dict(zip(OUT_NAMES, SESS.run(OUT_NAMES, {'input': tracks6[i:i + chunk]})))
        acc['per_track'].append(res[PER_TRACK_T])
        acc['mean'].append(res[MEAN_T])
        acc['out27'].append(res['output'])
    return {k: np.concatenate(v).astype(np.float64) for k, v in acc.items()}

print('session outputs:', OUT_NAMES)


In [ ]:
sys.path.insert(0, str(BATCHD))
import crosscheck as C     # numpy-only; an independent golden

X8  = load_txt('embed_input.txt').reshape(-1, 8)
x50 = X8[:TRACKS_PER_EVENT, :REAL_FEATURES][None, ...]        # (1, 50, 6)
r50 = run_onnx(x50)

emb_g, soln_g = C.golden()
np_per_track  = soln_g[2][:TRACKS_PER_EVENT]

d_track = np.abs(r50['per_track'][0] - np_per_track).max()
d_mean  = np.abs(r50['mean'][0] - np_per_track.mean(0)).max()
d_out   = np.abs(r50['out27'][0] - (np_per_track.mean(0) @ W['output'] + B['output'])).max()
print(f'  ONNX per-track  vs numpy golden : {d_track:.3e}')
print(f'  ONNX event mean vs numpy golden : {d_mean:.3e}')
print(f'  ONNX out27      vs numpy golden : {d_out:.3e}')

GATE = max(d_track, d_mean, d_out)
print(f'\n  worst = {GATE:.3e}   {"PASS" if GATE < 1e-6 else "FAIL"}')
assert GATE < 1e-6, 'the ONNX and the numpy golden disagree - resolve before continuing'


In [ ]:
ref_hw = C.L('aieml10_output_aie.txt').reshape(-1, H)[:TRACKS_PER_EVENT]
print(f'aieml/ VEK280 dump: {ref_hw.shape}  (per-track, pre-average)\n')
print(f'  per-track, tracks {WARMUP}..{TRACKS_PER_EVENT-1} : '
      f'{np.abs(r50["per_track"][0][WARMUP:] - ref_hw[WARMUP:]).max():.3e}   <- the real check')
print(f'  per-track, tracks 0..{WARMUP-1}        : '
      f'{np.abs(r50["per_track"][0][:WARMUP] - ref_hw[:WARMUP]).max():.3e}   <- warm-up, expected')
print(f'  its mean(axis=0) vs ONNX mean : '
      f'{np.abs(r50["mean"][0] - ref_hw.mean(0)).max():.3e}')

print('\nThe warm-up gap is definitional. The ONNX rolls CIRCULARLY inside a 50-track event'
      '\n(track 0 pairs with track 49). A streaming implementation cannot do that without'
      '\nbuffering the whole event, so both aieml/ and aieml_batch pair track 0 with zero (or'
      '\nwith the previous block). The receptive field is 4 tracks deep, so exactly tracks 0..2'
      '\ndiffer and 3..49 agree.')


In [ ]:
STIM = TESTD / 'embed_input_50000.txt'
assert STIM.exists(), f'{STIM} missing - run: cd aieml_batch && ./make_golden.py --tracks 50000 --out ../testdata'

raw = np.loadtxt(STIM).astype(np.float32).reshape(-1, 8)
N_TRACKS = len(raw)
assert N_TRACKS % TRACKS_PER_EVENT == 0, (
    f'{N_TRACKS} is not a multiple of {TRACKS_PER_EVENT}; track_accum counts SLOTS, so a short '
    'final event averages padding activations in and is wrong by design')
N_EVENTS = N_TRACKS // TRACKS_PER_EVENT

x_all = raw[:, :REAL_FEATURES].reshape(N_EVENTS, TRACKS_PER_EVENT, REAL_FEATURES)
print(f'{STIM.name}: {N_TRACKS} tracks -> {N_EVENTS} events')
print(f'  first 50 tracks are the real ones: {np.abs(raw[:50] - X8[:50]).max():.3e}  (0 = verbatim)')

REF_ALL = run_onnx(x_all)
for k, v in REF_ALL.items():
    print(f'  {k:10s} {v.shape}')


In [ ]:
chk_mean = np.abs(REF_ALL['per_track'].mean(axis=1) - REF_ALL['mean']).max()
chk_out  = np.abs(REF_ALL['mean'] @ W['output'] + B['output'] - REF_ALL['out27']).max()
print(f'  ReduceMean consistency                       : {chk_mean:.3e}')
print(f'  output Gemm, host convention (W as [128,27]) : {chk_out:.3e}')
assert chk_mean < 1e-5 and chk_out < 1e-4
print('\nInternally consistent. This is the reference.')

REF_NPZ = OUTD / f'golden_onnx_{N_TRACKS}.npz'
np.savez_compressed(
    REF_NPZ,
    per_track=REF_ALL['per_track'].astype(np.float32),   # (1000, 50, 128)
    mean=REF_ALL['mean'].astype(np.float32),             # (1000, 128) - what the AIE emits
    out27=REF_ALL['out27'].astype(np.float32),           # (1000, 27)  - after the host dense
    tracks=raw.astype(np.float32),
    n_events=N_EVENTS, n_tracks=N_TRACKS,
    source=str(ONNX_PATH.relative_to(REPO)), stimulus=STIM.name,
    warmup=WARMUP, tracks_per_event=TRACKS_PER_EVENT,
)
print(f'wrote {REF_NPZ.name}  ({REF_NPZ.stat().st_size/1e6:.1f} MB)')
print(f'  event 0 mean  range [{REF_ALL["mean"][0].min():.5f}, {REF_ALL["mean"][0].max():.5f}]')


In [ ]:
def load_sim(path):
    d   = np.load(path)
    nev = int(d['n_events'])
    spe = int(d['slots_per_event']) if 'slots_per_event' in d else SLOTS_PER_EVENT
    off = int(d['flush_slots']) if 'flush_slots' in d else 0
    out = {'file': path.name, 'sim': str(d['sim']) if 'sim' in d else '?',
           'n_events': nev, 'n_tracks': int(d['n_tracks']),
           'mean': d['measured'].astype(np.float64), 'tracks': d['tracks']}
    for k in ('emb_out', 's0_out', 's1_out', 's2_out'):
        if k in d:
            a = d[k].astype(np.float64)
            need = off + nev * spe
            if len(a) >= need:
                out[k] = a[off:need].reshape(nev, spe, H)[:, :TRACKS_PER_EVENT, :]
    return out

runs = sorted(SIMOUT.glob('sim_*ev_*tr.npz')) if SIMOUT.is_dir() else []
print(f'{len(runs)} run(s) in {SIMOUT}:')
for f in runs:
    d = np.load(f)
    per = [k for k in ('emb_out', 's0_out', 's1_out', 's2_out') if k in d]
    print(f'  {f.name:32s} {int(d["n_events"]):4d} ev, {int(d["n_tracks"]):6d} tr'
          f'   per-track: {"yes" if per else "NO (re-run to capture)"}')

# One run per simulator. Prefer per-track data (needed for §5), then the run matching
# TARGET_EVENTS, then the largest. Set TARGET_EVENTS = None to always take the largest.
TARGET_EVENTS = 5

RUNS = {}
for f in runs:
    d = np.load(f)
    tag = str(d['sim']) if 'sim' in d else ('x86' if '_x86_' in f.name else 'aie')
    nev = int(d['n_events'])
    key = ('s2_out' in d, TARGET_EVENTS is not None and nev == TARGET_EVENTS, nev)
    if tag not in RUNS or key > RUNS[tag][0]:
        RUNS[tag] = (key, f)
RUNS = {k: load_sim(v[1]) for k, v in sorted(RUNS.items())}
print(f'\ntarget {TARGET_EVENTS} events -> selected:',
      {k: v['file'] for k, v in RUNS.items()} or 'NONE')


In [ ]:
# Every run must have been fed a prefix of the same stimulus, or the comparison is meaningless.
for tag, sim in RUNS.items():
    n = min(sim['n_tracks'], N_TRACKS)
    d = np.abs(sim['tracks'][:n].astype(np.float32) - raw[:n]).max()
    sim['prefix_ok'] = d < 1e-6
    print(f'  {tag:3s}  {sim["n_tracks"]:6d} tracks   stimulus vs the 50k reference: {d:.3e}'
          f'   {"OK" if sim["prefix_ok"] else "MISMATCH"}')
    assert sim['prefix_ok'], f'{tag}: different stimulus - cannot compare against this reference'

def ref_for(sim):
    return {k: v[:sim['n_events']] for k, v in REF_ALL.items()}

print('\nevent mean over ALL 50 tracks, vs the ONNX reference:')
for tag, sim in RUNS.items():
    e = np.abs(sim['mean'] - ref_for(sim)['mean']).max(axis=1)
    print(f'  {tag:3s} ({sim["n_events"]:3d} ev)  worst {e.max():.3e}   median {np.median(e):.3e}')

if len(RUNS) == 2 and len(set(s['n_events'] for s in RUNS.values())) == 1:
    a, b = (RUNS[k] for k in RUNS)
    print(f'\n  {list(RUNS)[0]} vs {list(RUNS)[1]} event mean: '
          f'{np.abs(a["mean"] - b["mean"]).max():.3e}   <- simulator agreement')

print("""
EXPECT a few 1e-3 above, and do not read it as an error. This averages all 50 tracks including
the 3 warm-up ones, where the two designs legitimately differ. §5 separates the two causes.""")


In [ ]:
for tag, sim in RUNS.items():
    if 's2_out' not in sim:
        print(f'{tag}: no per-track data - re-run to capture it'); continue
    ref = ref_for(sim)
    aie = sim['s2_out']
    err = np.abs(aie - ref['per_track']).max(axis=2)      # (ev, 50)

    sim['per_track_err'] = err
    sim['m_all']  = np.abs(aie.mean(1) - ref['per_track'].mean(1)).max(axis=1)
    sim['m_warm'] = np.abs(aie[:, WARMUP:].mean(1) - ref['per_track'][:, WARMUP:].mean(1)).max(axis=1)

    print(f'--- {tag}  ({sim["n_events"]} events) ---')
    print('  per-track max|diff|, averaged over events:')
    for t in range(5):
        print(f'    track {t}: {err[:, t].mean():.3e}' + ('   <- warm-up' if t < WARMUP else ''))
    print(f'    tracks {WARMUP}..{TRACKS_PER_EVENT-1} : {err[:, WARMUP:].mean():.3e}'
          f'   (max {err[:, WARMUP:].max():.3e})')
    r = sim['m_all'].max() / max(sim['m_warm'].max(), 1e-30)
    print(f'  event mean, all {TRACKS_PER_EVENT} tracks : {sim["m_all"].max():.3e}')
    print(f'  event mean, tracks {WARMUP}..{TRACKS_PER_EVENT-1}  : {sim["m_warm"].max():.3e}   ({r:.0f}x better)\n')


In [ ]:
plot_runs = {k: v for k, v in RUNS.items() if 'per_track_err' in v}
if plot_runs:
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    colors = dict(zip(plot_runs, ['tab:blue', 'tab:orange']))

    for tag, sim in plot_runs.items():
        e, c = sim['per_track_err'], colors[tag]
        ax[0].fill_between(range(TRACKS_PER_EVENT), e.min(0), e.max(0), alpha=.2, color=c)
        ax[0].plot(e.mean(0), color=c, lw=1.5, label=f'{tag} (mean over events)')
        ax[1].plot(sim['m_all'],  'o-', ms=3, lw=1, color=c, label=f'{tag}: all {TRACKS_PER_EVENT}')
        ax[1].plot(sim['m_warm'], 's--', ms=3, lw=1, color=c, alpha=.7,
                   label=f'{tag}: tracks {WARMUP}..{TRACKS_PER_EVENT-1}')

    ax[0].axvspan(-0.5, WARMUP - 0.5, color='tab:red', alpha=.12)
    ax[0].text(WARMUP / 2, ax[0].get_ylim()[1], ' warm-up', color='tab:red',
               va='top', ha='center', fontsize=9)
    ax[0].set_yscale('log'); ax[0].set_xlabel('track index within event')
    ax[0].set_ylabel('max|AIE - ONNX|'); ax[0].set_title('(a) per-track agreement')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

    ax[1].set_yscale('log'); ax[1].set_xlabel('event')
    ax[1].set_ylabel('max|diff| of the event mean')
    ax[1].set_title('(b) event mean, both conventions')
    ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)

    allv = np.concatenate([np.r_[s['m_all'], s['m_warm']] for s in plot_runs.values()])
    bins = np.logspace(np.log10(allv.min() / 2), np.log10(allv.max() * 2), 25)
    for tag, sim in plot_runs.items():
        ax[2].hist(sim['m_all'],  bins=bins, alpha=.55, color=colors[tag], label=f'{tag}: all 50')
        ax[2].hist(sim['m_warm'], bins=bins, alpha=.55, color=colors[tag], histtype='step',
                   lw=2, label=f'{tag}: tracks {WARMUP}..49')
    ax[2].set_xscale('log'); ax[2].set_xlabel('max|diff| of the event mean')
    ax[2].set_ylabel('events'); ax[2].set_title('(c) distribution')
    ax[2].legend(fontsize=7); ax[2].grid(alpha=.3)

    fig.suptitle('aieml_batch vs the ONNX reference', y=1.02)
    fig.tight_layout()
    fig.savefig(OUTD / 'warmup_analysis.png', dpi=140, bbox_inches='tight')
    plt.show()


In [ ]:
if RUNS:
    tag, sim = next(iter(RUNS.items()))
    ref, ev = ref_for(sim), 0
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].plot(ref['mean'][ev], lw=1.2, label='ONNX reference')
    ax[0].plot(sim['mean'][ev], lw=1.0, ls='--', label=f'aieml_batch ({tag})')
    ax[0].set_xlabel('feature'); ax[0].set_ylabel('event mean')
    ax[0].set_title(f'(a) event {ev}: the 128-wide mean')
    ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    ax[1].plot(sim['mean'][ev] - ref['mean'][ev], lw=1, color='tab:red')
    ax[1].set_xlabel('feature'); ax[1].set_ylabel('AIE - ONNX')
    ax[1].set_title(f'(b) event {ev}: residual'); ax[1].grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(OUTD / 'event0_residual.png', dpi=140, bbox_inches='tight')
    plt.show()


In [ ]:
# Point this at the copied file OR the directory holding it. Accepts:
#   - track_means_all.txt        (single packed file - preferred)
#   - a directory containing it, or containing track_mean_128_ev*.txt
#   - a .npz with a 'means' or 'measured' array
HW = None    # e.g. HW = Path('/home/synthara/hw_results/track_means_all.txt')

def load_hw(where):
    """-> (n_events, 128) from whatever form the board data arrived in."""
    p = Path(where)
    if p.is_dir():
        packed = p / 'track_means_all.txt'
        if packed.exists():
            p = packed
        else:
            files = sorted(p.glob('track_mean_128_ev*.txt'),
                           key=lambda q: int(''.join(c for c in q.stem if c.isdigit()) or 0))
            if not files:
                raise FileNotFoundError(f'no track_means_all.txt or track_mean_128_ev*.txt in {p}')
            return np.stack([np.loadtxt(f) for f in files]), f'{len(files)} per-event files'
    if p.suffix == '.npz':
        d = np.load(p)
        key = 'means' if 'means' in d else 'measured'
        return np.asarray(d[key], dtype=np.float64), p.name
    # packed text: "<n_events> <n_cols>" header, then one row per event
    with p.open() as f:
        n_ev, n_col = (int(x) for x in f.readline().split())
    a = np.loadtxt(p, skiprows=1).reshape(n_ev, n_col)
    return a, p.name

if HW is None:
    print('HW not set - see the cell above. Nothing to compare yet.')
else:
    hw, src = load_hw(HW)
    n = min(len(hw), N_EVENTS)
    err = np.abs(hw[:n] - REF_ALL['mean'][:n]).max(axis=1)
    print(f'hardware: {hw.shape} from {src}')
    if len(hw) != N_EVENTS:
        print(f'  NOTE: {len(hw)} events vs {N_EVENTS} in the reference; comparing the first {n}')
    print(f'  worst     : {err.max():.3e}  (event {int(err.argmax())})')
    print(f'  median    : {np.median(err):.3e}')
    print(f'  over 1e-4 : {(err > 1e-4).sum()} of {n}')
    print("""
  Expect ~1e-2, the same as the simulations in §4: this is the event mean over all 50
  tracks, warm-up included. Compare it to the x86 number above - hardware and simulation
  should agree with each other far more closely than either agrees with the reference.""")

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
    ax[0].plot(err, 'o-', ms=2, lw=.7)
    ax[0].set_yscale('log'); ax[0].set_xlabel('event'); ax[0].set_ylabel('max|HW - ONNX|')
    ax[0].set_title('(a) per event'); ax[0].grid(alpha=.3)
    ax[1].hist(err, bins=40)
    ax[1].set_xlabel('max|HW - ONNX|'); ax[1].set_ylabel('events')
    ax[1].set_title('(b) distribution'); ax[1].grid(alpha=.3)
    fig.tight_layout(); fig.savefig(OUTD / 'hw_vs_reference.png', dpi=140, bbox_inches='tight')
    plt.show()

    for tag, sim in RUNS.items():
        m = min(len(hw), sim['n_events'])
        print(f'  hardware vs {tag} simulation, first {m} events: '
              f'{np.abs(hw[:m] - sim["mean"][:m]).max():.3e}')


In [ ]:
hw_tracks = np.array([50, 500, 5000, 10000], float)
hw_us     = np.array([635., 900., 3480., 6578.])
MACS_PER_TRACK, II_NS, ITER_PER_EVENT = 264192, 4161.0, 7

A = np.vstack([np.ones_like(hw_tracks), hw_tracks]).T
(fixed, marginal), *_ = np.linalg.lstsq(A, hw_us, rcond=None)
model_us = II_NS * 1e-3 * ITER_PER_EVENT * np.ceil(hw_tracks / TRACKS_PER_EVENT)
cycle_ns = II_NS * ITER_PER_EVENT / TRACKS_PER_EVENT

print(f'fit: total = {fixed:.1f} us + {marginal*1000:.1f} ns x tracks')
print(f'cycle-accurate marginal: {cycle_ns:.1f} ns/track'
      f'   (fit is {100*abs(marginal*1000-cycle_ns)/cycle_ns:.1f}% off)\n')
print(f'{"tracks":>7} {"measured":>11} {"fit":>10} {"AIE model":>11} {"overhead":>9}'
      f' {"ns/track":>9} {"GOP/s":>8}')
for t, y, f, m in zip(hw_tracks, hw_us, A @ [fixed, marginal], model_us):
    print(f'{t:7.0f} {y:9.1f}us {f:8.1f}us {m:9.1f}us {100*(1-m/y):8.1f}%'
          f' {y/t*1000:9.1f} {2*MACS_PER_TRACK*t/(y*1e-6)/1e9:8.1f}')

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
xs = np.linspace(1, hw_tracks.max() * 1.1, 200)
ax[0].plot(hw_tracks, hw_us, 'o', ms=7, label='measured')
ax[0].plot(xs, fixed + marginal * xs, '-', lw=1,
           label=f'{fixed:.0f} us + {marginal*1000:.0f} ns/track')
ax[0].plot(xs, II_NS*1e-3*ITER_PER_EVENT*np.ceil(xs/TRACKS_PER_EVENT), '--', lw=1,
           label='AIE compute alone')
ax[0].set_xlabel('tracks'); ax[0].set_ylabel('execute (us)')
ax[0].set_title('(a) total run time'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(hw_tracks, hw_us / hw_tracks * 1000, 'o-', ms=6, label='measured')
ax[1].axhline(marginal * 1000, ls='--', c='tab:green', lw=1, label=f'marginal {marginal*1000:.0f} ns')
ax[1].axhline(cycle_ns, ls=':', c='tab:red', lw=1, label=f'cycle-accurate {cycle_ns:.0f} ns')
ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].set_xlabel('tracks'); ax[1].set_ylabel('ns per track')
ax[1].set_title('(b) cost per track - the fixed cost amortising')
ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, which='both')
fig.tight_layout(); fig.savefig(OUTD / 'hw_performance.png', dpi=140, bbox_inches='tight')
plt.show()
